<a href="https://colab.research.google.com/github/hmabril-sketch/hig-reproducibility/blob/main/notebooks/HIG4_8_ed_20x20_Colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pintar y analizar HIG₄ y HIG₈ en 20 × 20

1. Ejecuta las tres celdas en orden en Google Colab.
2. Elige uno de los seis colores y pinta con clic o arrastrando el ratón. El color 0 llena inicialmente la imagen.
3. Pulsa **🔬 HIG analysis** para ver ambos grafos, los generadores tipados, sus tamaños y los dos recuentos de Euler.
4. Si quieres comparar dos pinturas, analiza la primera y pulsa **Guardar referencia A**; pinta la segunda y vuelve a analizar.

**Convención de tamaño.** Las regiones tienen área en píxeles; los $1$-generadores geométricos de ambos HIG tienen longitud en aristas primitivas. Los `cut` de $HIG_4$ muestran aristas de su soporte, que puede solaparse con otros soportes. Los nuevos `cut` de $HIG_8^{\mathrm{comb}}$ se indexan por agujeros $4$-conexos: mostramos el área del agujero asociado, sin afirmar que ese área sea soporte del generador. Los $0$-generadores virtuales y `OUT` carecen de área píxel propia.

La imagen es una **segmentación categórica de seis etiquetas**. Se comprueban las identidades $\chi_{\mathrm{gr}}(HIG_4)=2$, $\chi_{\mathrm{gr}}(HIG_8^{\mathrm{comb}})=2$ y $K_8-K_4=D-M$, contando `OUT` en ambos casos. La comparación A/B describe diferencias estructurales entre dibujos; no mide rendimiento predictivo sobre datos clínicos.

In [ ]:
from collections import Counter, defaultdict, deque
import numpy as np
from scipy import sparse

# Implementación autocontenida de la construcción intrínseca HIG₄.
# No necesita archivos auxiliares cuando se abre en Colab.

def region_ids(image):
    """Four-connected equal-label components; OUT receives the final id."""
    h, w = image.shape
    ids = np.full((h, w), -1, dtype=np.int32)
    nr = 0
    for y in range(h):
        for x in range(w):
            if ids[y, x] != -1:
                continue
            label = image[y, x]
            ids[y, x] = nr
            queue = deque([(y, x)])
            while queue:
                yy, xx = queue.popleft()
                for ny, nx in ((yy-1, xx), (yy+1, xx), (yy, xx-1), (yy, xx+1)):
                    if 0 <= ny < h and 0 <= nx < w and ids[ny, nx] == -1 and image[ny, nx] == label:
                        ids[ny, nx] = nr
                        queue.append((ny, nx))
            nr += 1
    return ids, nr


def construct(image):
    ids, out = region_ids(image)
    h, w = image.shape
    edges = []                 # (first grid vertex, second, region pair, on frame)
    at = defaultdict(list)
    def add(a, b, u, v, frame):
        if u == v:
            return
        i = len(edges)
        pair = tuple(sorted((int(u), int(v))))
        edges.append((a, b, pair, frame))
        at[a].append(i)
        at[b].append(i)
    for y in range(h):
        for x in range(w+1):
            add((y,x), (y+1,x), out if x == 0 else ids[y,x-1],
                out if x == w else ids[y,x], x in (0,w))
    for y in range(h+1):
        for x in range(w):
            add((y,x), (y,x+1), out if y == 0 else ids[y-1,x],
                out if y == h else ids[y,x], y in (0,h))
    junctions = sorted(v for v, incident in at.items() if len(incident) >= 3)
    jset = set(junctions)
    jindex = {v:i for i,v in enumerate(junctions)}
    geom = []
    seen = set()
    for start in range(len(edges)):
        if start in seen:
            continue
        queue = [start]
        seen.add(start)
        group = []
        for i in queue:
            group.append(i)
            a,b,pair,_ = edges[i]
            for v in (a,b):
                if v in jset:
                    continue
                for nxt in at[v]:
                    if nxt not in seen and edges[nxt][2] == pair:
                        seen.add(nxt)
                        queue.append(nxt)
        endpoints = []
        for i in group:
            for v in edges[i][:2]:
                if v in jset:
                    endpoints.append(v)
        if len(endpoints) == 0:
            typ = "enc0"
        elif len(endpoints) == 2:
            typ = "enc1" if endpoints[0] == endpoints[1] else "seg"
        else:
            raise ValueError(f"Continuation has {len(endpoints)} endpoint occurrences")
        geom.append({"type":typ,"pair":pair,"endpoints":endpoints,"edges":group})
    cuts = []
    seen = set()
    for start in range(len(edges)):
        if start in seen:
            continue
        queue = [start]
        seen.add(start)
        group = []
        frame = False
        for i in queue:
            group.append(i)
            a,b,_,on_frame = edges[i]
            frame |= on_frame
            for v in (a,b):
                for nxt in at[v]:
                    if nxt not in seen:
                        seen.add(nxt)
                        queue.append(nxt)
        if not frame:
            cuts.append(group)
    # Pixel-side to frontier edge; supports local validation on small images.
    pixel_sides = defaultdict(list)
    for i,(a,b,pair,_) in enumerate(edges):
        if a[0] == b[0]:
            yy=a[0]; xx=a[1]
            for py in (yy-1,yy):
                if 0<=py<h:pixel_sides[(py,xx)].append(i)
        else:
            yy=a[0]; xx=a[1]
            for px in (xx-1,xx):
                if 0<=px<w:pixel_sides[(yy,px)].append(i)
    # Each frame-free frontier component corresponds to exactly one
    # bounded 8-component of the complement of exactly one 4-region.
    # Establish the carrier by matching its complete R-boundary to the cut.
    # Its uppermost horizontal edge has the carrier pixel immediately above.
    # This gives a linear-time carrier extraction without flooding the
    # complement separately for every region (which is prohibitive for WSI).
    carriers=[]
    for group in cuts:
        horizontal=[i for i in group if edges[i][0][0]==edges[i][1][0]]
        i=min(horizontal,key=lambda k:(edges[k][0][0],edges[k][0][1]))
        yy,xx=edges[i][0]
        carriers.append(int(ids[yy-1,xx]))
    holes=Counter(carriers)
    if h*w<=10000:
      cut_sets = [set(g) for g in cuts]
      carrier_matches = [[] for _ in cuts]
      verified_holes=Counter()
      for region in range(out):
        visited = np.zeros((h,w), dtype=bool)
        for y in range(h):
            for x in range(w):
                if ids[y,x] == region or visited[y,x]:
                    continue
                queue = deque([(y,x)])
                visited[y,x] = True
                pixels = set()
                frame = False
                while queue:
                    yy,xx = queue.popleft()
                    pixels.add((yy,xx))
                    frame |= yy in (0,h-1) or xx in (0,w-1)
                    for dy in (-1,0,1):
                        for dx in (-1,0,1):
                            ny,nx = yy+dy,xx+dx
                            if 0 <= ny < h and 0 <= nx < w and not visited[ny,nx] and ids[ny,nx] != region:
                                visited[ny,nx] = True
                                queue.append((ny,nx))
                if frame:
                    continue
                verified_holes[region] += 1
                boundary = set()
                for py,px in pixels:
                    for i in pixel_sides[(py,px)]:
                        if region in edges[i][2]:boundary.add(i)
                # C can include frontiers between regions *inside* the hole;
                # only the R--H boundary must lie in that component.
                matches = [k for k,g in enumerate(cut_sets) if boundary and boundary <= g]
                if len(matches) != 1:
                    raise ValueError(f"Cut-hole correspondence failed at region {region}: {len(matches)} matches")
                carrier_matches[matches[0]].append(region)
      if any(m != [carriers[k]] for k,m in enumerate(carrier_matches)) or holes != verified_holes:
          raise ValueError("Fast carrier disagrees with explicit 8-hole verification")
    n0 = len(junctions) + sum(g["type"] == "enc0" for g in geom)
    n1 = len(geom) + len(cuts)
    n2 = out + 1
    ii01,jj01,vv01,ii12,jj12,vv12 = [],[],[],[],[],[]
    virtual = len(junctions)
    for k,g in enumerate(geom):
        endpoints = g["endpoints"]
        if g["type"] == "enc0":
            ii01.append(virtual); jj01.append(k); vv01.append(2)
            virtual += 1
        else:
            for j,count in Counter(jindex[v] for v in endpoints).items():
                ii01.append(j); jj01.append(k); vv01.append(count)
        for region in g["pair"]:
            ii12.append(k); jj12.append(region); vv12.append(1)
    for k,region in enumerate(carriers, start=len(geom)):
        ii12.append(k); jj12.append(region); vv12.append(2)
        # Intentionally NO 0--1 incidence for cut k.
    m01 = sparse.csr_matrix((vv01,(ii01,jj01)),shape=(n0,n1),dtype=np.int32)
    m12 = sparse.csr_matrix((vv12,(ii12,jj12)),shape=(n1,n2),dtype=np.int32)
    return ids, edges, geom, cuts, carriers, holes, m01, m12



In [ ]:

import html
from statistics import median
from scipy import ndimage

def _esc(value):
    return html.escape(str(value), quote=True)

def _table(headers, rows):
    head = ''.join(f'<th>{_esc(x)}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{_esc(x)}</td>' for x in row) + '</tr>' for row in rows)
    return f'<div class="hig-table-wrap"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

def _distribution(values):
    if not values:
        return '—'
    return f'{min(values)} / {median(values):g} / {max(values)}'

def _histogram(values):
    return [[int(value), int(count)] for value, count in sorted(Counter(values).items())]

def _derive_hig8(image, ids, geom, cuts, holes4, m01, m12, junctions):
    h, w = image.shape
    old_out = m12.shape[1] - 1
    parent = list(range(old_out))

    def root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        a, b = root(a), root(b)
        if a != b:
            parent[max(a, b)] = min(a, b)

    # q_v se lee sobre los cuatro cuadrantes completos de cada vértice interior.
    for y in range(1, h):
        for x in range(1, w):
            for (a, b) in (((y-1, x-1), (y, x)),
                           ((y-1, x), (y, x-1))):
                ra, rb = int(ids[a]), int(ids[b])
                if ra != rb and image[a] == image[b]:
                    union(ra, rb)

    roots = sorted({root(r) for r in range(old_out)})
    index = {r: i for i, r in enumerate(roots)}
    quotient = [index[root(r)] for r in range(old_out)]
    new_out = len(roots)
    quotient.append(new_out)    # OUT se transporta a OUT.
    qmap = np.asarray(quotient, dtype=int)[ids]
    junction_index = {v: i for i, v in enumerate(junctions)}
    diagonal = []
    d_by_q = Counter()
    for y in range(1, h):
        for x in range(1, w):
            nw, ne = int(qmap[y-1, x-1]), int(qmap[y-1, x])
            sw, se = int(qmap[y, x-1]), int(qmap[y, x])
            for qid, same, other_a, other_b, orientation in (
                (nw, se, ne, sw, 'NW–SE'),
                (ne, sw, nw, se, 'NE–SW')):
                if qid == same and other_a != qid and other_b != qid:
                    assert (y, x) in junction_index
                    diagonal.append({'vertex': (y, x), 'region': qid,
                                     'junction': junction_index[(y, x)],
                                     'orientation': orientation})
                    d_by_q[qid] += 1

    # Oráculo independiente para la agrupación: componentes 8 directas.
    direct = []
    for value in np.unique(image):
        labels, count = ndimage.label(image == value, structure=np.ones((3, 3), dtype=int))
        direct.extend((labels == i).tobytes() for i in range(1, count + 1))
    assert sorted(direct) == sorted((qmap == i).tobytes() for i in range(new_out))

    four_structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=int)
    hole_nodes = []
    regions = []
    for qid in range(new_out):
        mask = qmap == qid
        complement = np.pad(~mask, 1, constant_values=True)
        labels, count = ndimage.label(complement, structure=four_structure)
        outside = int(labels[0, 0])
        holes = [labels[1:-1, 1:-1] == i for i in range(1, count + 1) if i != outside]
        members = [r for r in range(old_out) if quotient[r] == qid]
        k = len(members)
        d = d_by_q[qid]
        h4 = sum(holes4.get(r, 0) for r in members)
        assert len(holes) == h4 + d - k + 1, (qid, k, d, h4, len(holes))
        coords = np.argwhere(mask)
        y, x = coords[0]
        regions.append({'id': qid, 'label': int(image[y, x]), 'pixels': len(coords),
                        'members': members, 'holes': len(holes), 'd': d,
                        'cycle_rank': d-k+1})
        for hole in holes:
            hole_nodes.append({'carrier': qid, 'hole_pixels': int(np.count_nonzero(hole)),
                               'first_pixel': tuple(int(c) for c in np.argwhere(hole)[0])})

    m = old_out - new_out
    d = len(diagonal)
    k4, k8 = len(cuts), len(hole_nodes)
    n0 = m01.shape[0] + d
    n1 = len(geom) + k8
    n2 = new_out + 1
    chi = n0 - n1 + n2
    assert k8-k4 == d-m
    assert chi == 2

    # Incidencias tipadas de la definición de HIG_8^{comb}.
    old_to_new = sparse.csr_matrix((np.ones(old_out+1, dtype=int),
                    (np.arange(old_out+1), quotient)), shape=(old_out+1, n2))
    retained12 = m12[:len(geom), :] @ old_to_new
    cut12 = sparse.csr_matrix((np.full(k8, 2, dtype=int),
                    (np.arange(k8), [item['carrier'] for item in hole_nodes])),
                    shape=(k8, n2))
    m12_8 = sparse.vstack([retained12, cut12], format='csr')
    old01 = m01[:, :len(geom)]
    new_rows = old01[[item['junction'] for item in diagonal], :]
    geometric01 = sparse.vstack([old01, new_rows], format='csr')
    m01_8 = sparse.hstack([geometric01, sparse.csr_matrix((n0, k8), dtype=int)], format='csr')
    assert m01_8.shape == (n0, n1) and m12_8.shape == (n1, n2)
    assert m01_8[:, len(geom):].nnz == 0
    assert all(m12_8[len(geom)+i, item['carrier']] == 2
               for i, item in enumerate(hole_nodes))

    return {'regions': regions, 'diagonal': diagonal, 'holes': hole_nodes,
            'quotient': quotient, 'm': m, 'd': d, 'k4': k4, 'k8': k8,
            'n0': n0, 'n1': n1, 'n2': n2, 'chi': chi,
            'm01_total': int(m01_8.sum()), 'm12_total': int(m12_8.sum())}

def _analyse_hig4(labels):
    if not isinstance(labels, list) or len(labels) != 400:
        raise ValueError('Se necesitan exactamente 400 etiquetas.')
    if any(type(x) is not int or not 0 <= x <= 5 for x in labels):
        raise ValueError('Cada píxel debe tener una etiqueta entera entre 0 y 5.')
    image = np.asarray(labels, dtype=np.int32).reshape(20, 20)
    ids, edges, geom, cuts, carriers, holes, m01, m12 = construct(image)
    n0, n1 = m01.shape
    n2 = m12.shape[1]
    out = n2 - 1
    chi = n0 - n1 + n2
    counts = Counter(g['type'] for g in geom)
    degree = Counter(v for a, b, _, _ in edges for v in (a, b))
    junctions = sorted(v for v, d in degree.items() if d >= 3)
    virtual = [(i, g) for i, g in enumerate(geom) if g['type'] == 'enc0']
    assert n0 == len(junctions) + len(virtual)
    assert n1 == len(geom) + len(cuts)
    assert chi == 2
    assert int(m01[:, len(geom):].nnz) == 0
    assert sum(holes.values()) == len(cuts)

    groups = [
        ('V₀ junction', len(junctions), [1] * len(junctions)),
        ('V₀ enclosing virtual', len(virtual), []),
        ('V₁ segmentation', counts['seg'], [len(g['edges']) for g in geom if g['type'] == 'seg']),
        ('V₁ enclosing, loop', counts['enc1'], [len(g['edges']) for g in geom if g['type'] == 'enc1']),
        ('V₁ enclosing, closed cycle', counts['enc0'], [len(g['edges']) for g in geom if g['type'] == 'enc0']),
        ('V₁ cut', len(cuts), [len(c) for c in cuts]),
        ('V₂ image region', out, np.bincount(ids.ravel(), minlength=out).tolist()),
        ('V₂ OUT', 1, []),
    ]
    summary_rows = [(name, cardinal, _distribution(sizes)) for name, cardinal, sizes in groups]

    zero_rows = []
    for i, v in enumerate(junctions):
        zero_rows.append((f'v₀:{i}', 'junction', 1, f'({v[0]}, {v[1]})', degree[v]))
    for j, (geom_id, g) in enumerate(virtual, start=len(junctions)):
        zero_rows.append((f'v₀:{j}', 'enclosing virtual', '—', f'ciclo v₁:{geom_id}', len(g['edges'])))

    one_rows = []
    for i, g in enumerate(geom):
        one_rows.append((f'v₁:{i}', g['type'], len(g['edges']),
                         str(g['pair']), str(g['endpoints'])))
    for j, (component, carrier) in enumerate(zip(cuts, carriers), start=len(geom)):
        one_rows.append((f'v₁:{j}', 'cut', len(component),
                         f'portador v₂:{carrier}', 'sin incidencia 0–1'))

    two_rows = []
    for i in range(out):
        loc = np.argwhere(ids == i)
        y, x = loc[0]
        two_rows.append((f'v₂:{i}', int(image[y, x]), len(loc), f'({y}, {x})'))
    two_rows.append((f'v₂:{out}', 'OUT', '—', 'exterior compactificado'))

    hig8 = _derive_hig8(image, ids, geom, cuts, holes, m01, m12, junctions)
    qregions, occurrences, new_holes = hig8['regions'], hig8['diagonal'], hig8['holes']
    eight_groups = [
        ('V₀ junction (retenidos)', len(junctions), [1] * len(junctions)),
        ('V₀ enclosing virtual (retenidos)', len(virtual), []),
        ('V₀ diagonal tipado z', len(occurrences), [1] * len(occurrences)),
        ('V₁ segmentation (retenidos)', counts['seg'], [len(g['edges']) for g in geom if g['type'] == 'seg']),
        ('V₁ enclosing loop (retenidos)', counts['enc1'], [len(g['edges']) for g in geom if g['type'] == 'enc1']),
        ('V₁ enclosing ciclo (retenidos)', counts['enc0'], [len(g['edges']) for g in geom if g['type'] == 'enc0']),
        ('V₁ cut: área del agujero asociado', len(new_holes), [item['hole_pixels'] for item in new_holes]),
        ('V₂ región 8-conexa', len(qregions), [r['pixels'] for r in qregions]),
        ('V₂ OUT', 1, []),
    ]
    eight_summary = [(name, cardinal, _distribution(sizes)) for name, cardinal, sizes in eight_groups]
    diag_rows = [(f'z:{n0+i}', f'({z["vertex"][0]}, {z["vertex"][1]})',
                  f'Q:{z["region"]}', z['orientation'], f'junction v₀:{z["junction"]}')
                 for i, z in enumerate(occurrences)]
    hole_rows = [(f'c:{len(geom)+i}', f'Q:{item["carrier"]}', item['hole_pixels'],
                  str(item['first_pixel']), 'incidencia 1–2 = 2; columna 0–1 = 0')
                 for i, item in enumerate(new_holes)]
    region_rows = [(f'Q:{r["id"]}', r['label'], r['pixels'], str(r['members']),
                    r['holes'], r['d'], r['cycle_rank']) for r in qregions]

    markup = f'''
    <div class="hig-analysis">
      <div class="hig-cards">
        <div><strong>|V₀|</strong><span>{n0}</span></div>
        <div><strong>|V₁|</strong><span>{n1}</span></div>
        <div><strong>|V₂| (incluye OUT)</strong><span>{n2}</span></div>
        <div><strong>χ = |V₀| − |V₁| + |V₂|</strong><span>{n0} − {n1} + {n2} = {chi}</span></div>
      </div>
      <h3>Cardinales y tamaños del soporte</h3>
      <p>En cada fila: mínimo / mediana / máximo. «—» indica ausencia de tamaño geométrico propio.</p>
      {_table(('Tipo', 'Cardinal', 'Tamaño mín. / med. / máx.'), summary_rows)}
      <details><summary>V₀ · {n0} nodos (ver detalles)</summary>
        {_table(('Nodo', 'Tipo', 'Tamaño', 'Ubicación o ciclo asociado', 'Grado o longitud'), zero_rows)}
      </details>
      <details><summary>V₁ · {n1} nodos (ver detalles)</summary>
        {_table(('Nodo', 'Tipo', 'Aristas del soporte', 'Regiones o portador', 'Extremos'), one_rows)}
      </details>
      <details><summary>V₂ · {n2} nodos (ver detalles)</summary>
        {_table(('Nodo', 'Etiqueta', 'Píxeles', 'Píxel inicial / descripción'), two_rows)}
      </details>
      <p class="hig-foot">Verificado: χ = 2; {len(cuts)} cuts = {sum(holes.values())} agujeros
      (4-región / 8-complemento); columnas 0–1 de los cuts nulas.</p>
    </div>'''
    markup += f'''
    <div class="hig-analysis">
      <h2>HIG₈ combinatorio derivado</h2>
      <div class="hig-cards">
        <div><strong>|V₀⁽⁸⁾|</strong><span>{hig8['n0']}</span></div>
        <div><strong>|V₁⁽⁸⁾|</strong><span>{hig8['n1']}</span></div>
        <div><strong>|V₂⁽⁸⁾| (incluye OUT)</strong><span>{hig8['n2']}</span></div>
        <div><strong>χ₈</strong><span>{hig8['n0']} − {hig8['n1']} + {hig8['n2']} = {hig8['chi']}</span></div>
      </div>
      <p>Quotient: M = {hig8['m']} regiones fusionadas; D = {hig8['d']}
      ocurrencias diagonales; K₄ = {hig8['k4']}; K₈ = {hig8['k8']}.
      Verificado: K₈ − K₄ = {hig8['k8']-hig8['k4']} = D − M.</p>
      <p>Los (1)-generadores geométricos conservan su tamaño en aristas;
      para los nuevos cuts se indica el área del agujero complementario que los indexa.</p>
      {_table(('Tipo', 'Cardinal', 'Tamaño mín. / med. / máx.'), eight_summary)}
      <details><summary>V₀: {len(occurrences)} nuevos nodos diagonales tipados</summary>
        {_table(('Nodo', 'Vértice', 'Región 8', 'Diagonal', 'Fila 0–1 duplicada de'), diag_rows)}
      </details>
      <details><summary>V₁: {len(new_holes)} nuevos cuts por agujeros 4-conexos</summary>
        {_table(('Nodo', 'Portador', 'Píxeles del agujero', 'Primer píxel', 'Incidencia'), hole_rows)}
      </details>
      <details><summary>V₂: {len(qregions)} regiones 8-conexas y OUT</summary>
        {_table(('Nodo', 'Etiqueta', 'Píxeles', 'Regiones 4 constituyentes',
                 'Agujeros 4', 'd', 'd − k + 1'), region_rows)}
        <p>OUT es el generador adicional, sin área píxel.</p>
      </details>
      <p class="hig-foot">Incidencias reconstruidas: Σμ₀,₁ = {hig8['m01_total']},
      Σμ₁,₂ = {hig8['m12_total']}. Las filas diagonales duplican las de su junction
      y las columnas de los nuevos cuts en μ₀,₁ son nulas.</p>
    </div>'''
    area4 = np.bincount(ids.ravel(), minlength=out).tolist()
    features = {
        'HIG₄ · junctions': len(junctions),
        'HIG₄ · enclosing 0': len(virtual),
        'HIG₄ · segmentos 1': counts['seg'],
        'HIG₄ · enclosing 1 tipo 0 / tipo 1': [counts['enc0'], counts['enc1']],
        'HIG₄ · cuts': len(cuts),
        'HIG₄ · regiones 4': out,
        'HIG₄ · distribución de áreas regionales': _histogram(area4),
        'HIG₄ · distribución de longitudes de contacto': _histogram([len(g['edges']) for g in geom]),
        'HIG₈ · regiones 8': len(qregions),
        'HIG₈ · fusiones M': hig8['m'],
        'HIG₈ · ocurrencias diagonales D': hig8['d'],
        'HIG₈ · cuts por agujeros 4': hig8['k8'],
        'HIG₈ · distribución de áreas regionales': _histogram([r['pixels'] for r in qregions]),
        'HIG₈ · agujeros por región': _histogram([r['holes'] for r in qregions]),
        'HIG₈ · distribución de áreas de agujeros': _histogram([item['hole_pixels'] for item in new_holes]),
        'HIG₈ · ciclos de contacto diagonal': sum(r['cycle_rank'] for r in qregions),
    }
    return {'html': markup, 'features': features,
            'counts': {'hig4': {'v0': n0, 'v1': n1, 'v2': n2, 'chi': chi, 'cuts': len(cuts)},
                       'hig8': {'v0': hig8['n0'], 'v1': hig8['n1'], 'v2': hig8['n2'],
                                'chi': hig8['chi'], 'cuts': hig8['k8'],
                                'diagonal': hig8['d'], 'merged': hig8['m']}}}

# El botón del lienzo invoca esta función en el kernel de Colab.
from IPython.display import JSON
from google.colab import output
def _colab_hig_callback(labels):
    return JSON(_analyse_hig4(labels))
output.register_callback('hig20.analyze', _colab_hig_callback)
print('Motores HIG₄ y HIG₈ combinatorio listos: tamaños, incidencias e identidades.')


Motores HIG₄ y HIG₈ combinatorio listos: tamaños, incidencias e identidades.


In [ ]:

from IPython.display import HTML, display
from uuid import uuid4
import json

widget_id = 'hig20-' + uuid4().hex
style = """
<style>
.hig20 {font-family:system-ui,sans-serif;max-width:1050px;color:#17212b}
.hig20 .toolbar{display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin:10px 0}
.hig20 button{border:1px solid #a6b2bd;border-radius:8px;background:white;padding:8px 11px;cursor:pointer;font-weight:600}
.hig20 button:hover{background:#edf3f7}.hig20 button.selected{outline:3px solid #27384e;outline-offset:1px}
.hig20 .analyse{background:#173b64;color:white;border-color:#173b64}
.hig20 .analyse:hover{background:#28557f;color:white}
.hig20 canvas{width:min(100%,520px);height:auto;touch-action:none;cursor:crosshair;border:2px solid #263746;border-radius:6px;image-rendering:pixelated}
.hig20 .hig-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(145px,1fr));gap:8px;margin:12px 0}
.hig20 .hig-cards>div{padding:12px;border:1px solid #cad6df;background:#f4f8fa;border-radius:8px}
.hig20 .hig-cards strong,.hig20 .hig-cards span{display:block}.hig20 .hig-cards span{font-size:20px;margin-top:5px}
.hig20 table{border-collapse:collapse;width:100%;font-size:13px}
.hig20 td,.hig20 th{padding:5px 7px;border-bottom:1px solid #dce4e9;text-align:left;white-space:nowrap}
.hig20 th{background:#f0f4f7}.hig20 .hig-table-wrap{max-height:370px;overflow:auto;margin:7px 0 15px}
.hig20 details{margin:8px 0;border:1px solid #dce4e9;padding:8px;border-radius:8px}
.hig20 summary{cursor:pointer;font-weight:600}.hig20 .hig-foot{font-size:13px;color:#405364}
</style>
"""
markup = f"""
<div id="{widget_id}" class="hig20">
 <h2>Editor HIG₄ / HIG₈ · 20 × 20</h2>
 <p>Elige una etiqueta y pinta con el ratón o el dedo. Cada casilla es un píxel.</p>
 <div class="toolbar" data-brushes></div>
 <canvas width="520" height="520" aria-label="Lienzo de 20 por 20 píxeles"></canvas>
 <div class="toolbar">
  <button class="analyse" data-analyse>🔬 HIG analysis</button>
  <button data-save-a>Guardar referencia A</button>
  <button data-clear>Limpiar (color 0)</button>
  <button data-demo>Ejemplo con encierro</button>
  <span data-status role="status">Pinta la imagen y pulsa HIG analysis.</span>
 </div>
 <div data-compare></div>
 <div data-results></div>
</div>"""

js = r"""
(() => {
 const root = document.getElementById(WIDGET_ID);
 const canvas = root.querySelector('canvas');
 const ctx = canvas.getContext('2d');
 const colors = ['#3478b7','#f08328','#3da65b','#9a60be','#d64f60','#e6c648'];
 const pixels = Array(400).fill(0);
 const brushes = root.querySelector('[data-brushes]');
 const status = root.querySelector('[data-status]');
 const results = root.querySelector('[data-results]');
 const comparison = root.querySelector('[data-compare]');
 let brush = 0, dragging = false, revision = 0, last = null, baseline = null;
 colors.forEach((color, index) => {
   const button = document.createElement('button');
   button.textContent = `Color ${index}`;
   button.title = `Etiqueta ${index}`;
   button.style.borderLeft = `12px solid ${color}`;
   button.addEventListener('click', () => {
     brush = index;
     [...brushes.querySelectorAll('button')].forEach((b,i) => b.classList.toggle('selected',i===index));
   });
   brushes.appendChild(button);
 });
 brushes.querySelector('button').classList.add('selected');
 function draw() {
   for (let y=0;y<20;y++) for (let x=0;x<20;x++) {
     ctx.fillStyle = colors[pixels[y*20+x]];
     ctx.fillRect(x*26,y*26,26,26);
   }
   ctx.strokeStyle = 'rgba(12,29,48,.32)';ctx.lineWidth=1;
   for(let i=0;i<=20;i++){
     ctx.beginPath();ctx.moveTo(i*26+.5,0);ctx.lineTo(i*26+.5,520);ctx.stroke();
     ctx.beginPath();ctx.moveTo(0,i*26+.5);ctx.lineTo(520,i*26+.5);ctx.stroke();
   }
 }
 function paint(event) {
   const box=canvas.getBoundingClientRect();
   const x=Math.floor((event.clientX-box.left)*20/box.width);
   const y=Math.floor((event.clientY-box.top)*20/box.height);
   if(x<0||x>=20||y<0||y>=20)return;
   const i=y*20+x;
   if(pixels[i]!==brush){pixels[i]=brush;revision++;draw();}
   status.textContent=`Píxel (${y}, ${x}): etiqueta ${pixels[i]}`;
 }
 canvas.addEventListener('pointerdown',e=>{dragging=true;canvas.setPointerCapture(e.pointerId);paint(e);});
 canvas.addEventListener('pointermove',e=>{if(dragging)paint(e);});
 canvas.addEventListener('pointerup',()=>{dragging=false;});
 canvas.addEventListener('pointercancel',()=>{dragging=false;});
 root.querySelector('[data-clear]').addEventListener('click',()=>{
   pixels.fill(0);revision++;last=null;draw();results.innerHTML='';comparison.innerHTML='';status.textContent='Lienzo limpio.';
 });
 root.querySelector('[data-demo]').addEventListener('click',()=>{
   pixels.fill(0);
   for(let y=3;y<=16;y++)for(let x=3;x<=16;x++)pixels[y*20+x]=1;
   for(let y=6;y<=13;y++)for(let x=6;x<=13;x++)pixels[y*20+x]=2;
   for(let y=9;y<=10;y++)for(let x=9;x<=10;x++)pixels[y*20+x]=3;
   revision++;last=null;draw();results.innerHTML='';comparison.innerHTML='';status.textContent='Ejemplo cargado: pulsa HIG analysis.';
 });
 root.querySelector('[data-save-a]').addEventListener('click',()=>{
   if(!last || last.revision!==revision){status.textContent='Analiza la imagen actual antes de guardarla como A.';return;}
   baseline={pixels:pixels.slice(),features:structuredClone(last.features)};
   comparison.innerHTML='';status.textContent='Referencia A guardada. Pinta B y vuelve a analizar.';
 });
 function showComparison(features) {
   comparison.innerHTML='';
   if(!baseline)return;
   const changed=Object.keys(features).filter(key=>
     JSON.stringify(features[key])!==JSON.stringify(baseline.features[key]));
   const differingPixels=pixels.filter((v,i)=>v!==baseline.pixels[i]).length;
   const title=document.createElement('h3');title.textContent=`Comparación A/B · ${differingPixels} píxeles con etiqueta distinta`;
   comparison.appendChild(title);
   const note=document.createElement('p');
   const newEight=changed.filter(key=>key.startsWith('HIG₈'));
   note.textContent=changed.length
     ? `${changed.length} descriptores agregados cambian; ${newEight.length} pertenecen a HIG₈.`
     : 'Las firmas agregadas coinciden; eso no demuestra que las imágenes sean idénticas.';
   comparison.appendChild(note);
   if(changed.length){
     const container=document.createElement('div');container.className='hig-table-wrap';
     const table=document.createElement('table');
     const head=document.createElement('thead');const header=document.createElement('tr');
     ['Descriptor','A','B'].forEach(text=>{const th=document.createElement('th');th.textContent=text;header.appendChild(th)});
     head.appendChild(header);table.appendChild(head);
     const body=document.createElement('tbody');
     changed.forEach(key=>{
       const row=document.createElement('tr');
       [key,JSON.stringify(baseline.features[key]),JSON.stringify(features[key])].forEach(text=>{
         const cell=document.createElement('td');cell.textContent=text;row.appendChild(cell);
       });body.appendChild(row);
     });
     table.appendChild(body);container.appendChild(table);comparison.appendChild(container);
   }
 }
 const analyse=root.querySelector('[data-analyse]');
 analyse.addEventListener('click',async()=>{
   analyse.disabled=true;status.textContent='Calculando HIG₄ y HIG₈…';
   const snapshot=revision;
   try {
     const reply=await google.colab.kernel.invokeFunction('hig20.analyze',[pixels.slice()],{});
     const raw=reply.data?.['application/json'] ?? reply.data?.['text/plain'];
     const data=typeof raw==='string'?JSON.parse(raw):raw;
     if(!data || !data.html)throw new Error('Respuesta del kernel sin resultados. Ejecuta de nuevo la celda del motor.');
     results.innerHTML=data.html;
     last={revision:snapshot,features:data.features};
     if(snapshot===revision)showComparison(data.features);
     status.textContent=snapshot===revision?'Análisis completado.':'Análisis completado para la imagen anterior: vuelve a pulsar para actualizar.';
   } catch(err){status.textContent='Error: '+String(err.message||err);}
   finally{analyse.disabled=false;}
 });
 draw();
})();
""".replace('WIDGET_ID', json.dumps(widget_id))
display(HTML(style + markup + '<script>' + js + '</script>'))


## Lectura de los resultados

- **$HIG_4$**: $V_0^{\mathrm{junc}}$ ocupa un vértice; $V_0^{\mathrm{enc}}$ es virtual. Los segmentos y ciclos de $V_1^{\mathrm{seg}}\sqcup V_1^{\mathrm{enc}}$ se miden en aristas; el soporte de un $V_1^{\mathrm{cut}}$ también se mide en aristas y puede solaparse con los anteriores. Las $V_2^{\mathrm{img}}$ son regiones $4$-conexas medidas en píxeles.
- **$HIG_8^{\mathrm{comb}}$**: se conservan los generadores geométricos; cada ocurrencia diagonal $(v,\mathcal R)$ crea un $0$-generador tipado $z_{v,\mathcal R}$. Cada agujero $4$-conexo del complemento de una región $8$-conexa crea un nuevo `cut`, cuyo tamaño asociado mostrado es el área del agujero. Las $V_2^{(8)}$ se miden en píxeles.
- **`OUT`**: se incluye en ambos cardinales de $V_2$ y no tiene área finita. Los dos valores de Euler son **recuentos alternados de generadores tipados**; el segundo no afirma una nueva homología ni un complejo celular plano regular.

El botón vuelve a construir las dos representaciones sobre la imagen actual y comprueba las identidades locales y globales. La comparación A/B enumera los descriptores que cambian; firmas agregadas idénticas no garantizan imágenes idénticas. No utiliza un clasificador entrenado.